# Notebook 03 — Feature Engineering
## Telecom Customer Churn and Retention Intelligence Dashboard

---

**Purpose:** Transform the cleaned staging dataset into an enriched analytical dataset by engineering high-value business features, behavioral flags, tenure cohorts, charge tiers, and service adoption counts.

**Input:** `data/staging/telco_staging.csv`

**Output:** `data/processed/telco_cleaned.csv`

**Key Feature Groups to Engineer:**
1. **Numeric Churn Flag:** `churn_flag` (1 = Churned, 0 = Retained) for mathematical aggregation
2. **Tenure Cohorts:** `tenure_cohort` (`0-12m`, `13-24m`, `25-48m`, `49-72m`)
3. **Charges Tiers:** `monthly_charges_tier` (`Low (<$35)`, `Medium ($35-$70)`, `High ($70-$90)`, `Very High (>$90)`)
4. **Service Adoption Metrics:** `total_services_subscribed`, `has_internet`, `has_phone`, `has_streaming`
5. **Tech Support / Security Protection Score:** `security_tech_score`
6. **Payment & Billing Profiles:** `payment_type_grouped`, `auto_payment_flag`
7. **Family / Demographic Status:** `family_status`
8. **Risk Indicators:** `high_risk_contract`, `paperless_echeck_risk`

---

## Section 1 — Import Libraries and Load Staging Data

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

PROJECT_ROOT = Path.cwd().parent
STAGING_FILE = PROJECT_ROOT / 'data' / 'staging' / 'telco_staging.csv'

df = pd.read_csv(STAGING_FILE)
print(f'Staging data loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')

## Section 2 — Numeric Churn Flag (`churn_flag`)

**Why:** Having a binary 1/0 indicator allows seamless calculation of churn rates (`AVG(churn_flag)` in SQL or `AVERAGE(churn_flag)` in DAX) without writing cumbersome `IF/SWITCH` statements.

In [ ]:
df['churn_flag'] = np.where(df['churn'] == 'Yes', 1, 0)
print(f'Churn Flag Distribution:')
display(df['churn_flag'].value_counts(normalize=True).mul(100).round(2))

## Section 3 — Tenure Cohorts (`tenure_cohort`)

**Why:** Churn behavior changes drastically over customer lifecycle:
- `0-12 Months (Year 1)`: Onboarding / high attrition danger zone
- `13-24 Months (Year 2)`: Early stabilization
- `25-48 Months (Years 3-4)`: Established customers
- `49-72 Months (Years 5+)`: Long-term loyal advocates

In [ ]:
bins_tenure = [-1, 12, 24, 48, 72]
labels_tenure = ['0-12m (Year 1)', '13-24m (Year 2)', '25-48m (Years 3-4)', '49-72m (Years 5+)']

df['tenure_cohort'] = pd.cut(df['tenure'], bins=bins_tenure, labels=labels_tenure)

print('Tenure Cohort Distribution & Churn Rates:')
cohort_analysis = df.groupby('tenure_cohort', observed=False).agg(
    total_customers=('customer_id', 'count'),
    churned_customers=('churn_flag', 'sum'),
    churn_rate=('churn_flag', lambda x: round(x.mean() * 100, 2))
).reset_index()
display(cohort_analysis)

## Section 4 — Monthly Charges Tiers (`monthly_charges_tier`)

**Why:** Grouping pricing into discrete tiers helps business managers analyze price elasticity and identify price sensitivity thresholds.

In [ ]:
bins_charges = [0, 35, 70, 90, 150]
labels_charges = ['Low (<$35)', 'Medium ($35-$70)', 'High ($70-$90)', 'Very High (>$90)']

df['monthly_charges_tier'] = pd.cut(df['monthly_charges'], bins=bins_charges, labels=labels_charges)

print('Monthly Charges Tier Distribution & Churn Rates:')
tier_analysis = df.groupby('monthly_charges_tier', observed=False).agg(
    total_customers=('customer_id', 'count'),
    churned_customers=('churn_flag', 'sum'),
    churn_rate=('churn_flag', lambda x: round(x.mean() * 100, 2))
).reset_index()
display(tier_analysis)

## Section 5 — Service Adoption & Count Engineering

**Why:** Customers with more products have higher switching barriers and lower churn. We count the active subscriptions across all 9 possible service lines.

In [ ]:
# Individual service flags (1 if active 'Yes', 0 otherwise)
df['has_phone'] = np.where(df['phone_service'] == 'Yes', 1, 0)
df['has_multiple_lines'] = np.where(df['multiple_lines'] == 'Yes', 1, 0)
df['has_internet'] = np.where(df['internet_service'] != 'No', 1, 0)
df['has_online_security'] = np.where(df['online_security'] == 'Yes', 1, 0)
df['has_online_backup'] = np.where(df['online_backup'] == 'Yes', 1, 0)
df['has_device_protection'] = np.where(df['device_protection'] == 'Yes', 1, 0)
df['has_tech_support'] = np.where(df['tech_support'] == 'Yes', 1, 0)
df['has_streaming_tv'] = np.where(df['streaming_tv'] == 'Yes', 1, 0)
df['has_streaming_movies'] = np.where(df['streaming_movies'] == 'Yes', 1, 0)

# Total services subscribed (sum across all 9 services)
service_cols = [
    'has_phone', 'has_multiple_lines', 'has_internet',
    'has_online_security', 'has_online_backup', 'has_device_protection',
    'has_tech_support', 'has_streaming_tv', 'has_streaming_movies'
]

df['total_services_subscribed'] = df[service_cols].sum(axis=1)

# Premium tech/security protection score (Online Security + Tech Support + Online Backup + Device Protection)
df['security_tech_score'] = df[['has_online_security', 'has_tech_support', 'has_online_backup', 'has_device_protection']].sum(axis=1)

# Entertainment bundle flag (Streaming TV + Streaming Movies)
df['has_streaming_bundle'] = np.where((df['has_streaming_tv'] == 1) & (df['has_streaming_movies'] == 1), 1, 0)

print('Service Count vs Churn Rate:')
service_summary = df.groupby('total_services_subscribed').agg(
    customers=('customer_id', 'count'),
    churn_rate=('churn_flag', lambda x: round(x.mean() * 100, 2))
).reset_index()
display(service_summary)

## Section 6 — Payment & Billing Risk Flags

**Why:** Electronic checks and paperless billing without autopay have historically shown 40%+ churn rates in telecom portfolios.

In [ ]:
# Autopay flag
df['auto_payment_flag'] = np.where(
    df['payment_method'].str.contains('automatic', case=False), 1, 0
)

# High risk payment combination (Paperless Billing + Electronic Check)
df['paperless_echeck_risk'] = np.where(
    (df['paperless_billing'] == 'Yes') & (df['payment_method'] == 'Electronic check'), 1, 0
)

# High risk contract (Month-to-month)
df['high_risk_contract'] = np.where(df['contract'] == 'Month-to-month', 1, 0)

# Simplified payment category
df['payment_type_grouped'] = df['payment_method'].replace({
    'Bank transfer (automatic)': 'Automatic (Bank/Card)',
    'Credit card (automatic)': 'Automatic (Bank/Card)',
    'Electronic check': 'Electronic check',
    'Mailed check': 'Mailed check'
})

print('Payment Method Grouped vs Churn Rate:')
display(df.groupby('payment_type_grouped').agg(
    customers=('customer_id', 'count'),
    churn_rate=('churn_flag', lambda x: round(x.mean() * 100, 2))
).reset_index())

## Section 7 — Family / Demographic Status Feature

**Why:** Household dynamics (having partner & dependents) significantly impact churn elasticity.

In [ ]:
def classify_family(row):
    if row['partner'] == 'Yes' and row['dependents'] == 'Yes':
        return 'Family (Partner & Dependents)'
    elif row['partner'] == 'Yes' and row['dependents'] == 'No':
        return 'Partner Only'
    elif row['partner'] == 'No' and row['dependents'] == 'Yes':
        return 'Single Parent'
    else:
        return 'Solo / Single'

df['family_status'] = df.apply(classify_family, axis=1)

print('Family Status Distribution & Churn Rate:')
display(df.groupby('family_status').agg(
    customers=('customer_id', 'count'),
    churn_rate=('churn_flag', lambda x: round(x.mean() * 100, 2))
).reset_index())

## Section 8 — Save Enriched Processed Dataset

**Why:** Export the final cleaned and feature-engineered dataset to `data/processed/telco_cleaned.csv` for Phase 6 segmentation, SQL ingestion, and Power BI modeling.

In [ ]:
processed_dir = PROJECT_ROOT / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

output_file = processed_dir / 'telco_cleaned.csv'
df.to_csv(output_file, index=False)

print(f'Processed dataset saved successfully to:')
print(f'  {output_file}')
print(f'Final Processed Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Engineered {df.shape[1] - 21} new features.')

## Section 9 — Summary of Engineered Features

In [ ]:
new_features_summary = pd.DataFrame({
    'Feature Name': [
        'churn_flag',
        'tenure_cohort',
        'monthly_charges_tier',
        'total_services_subscribed',
        'security_tech_score',
        'has_streaming_bundle',
        'auto_payment_flag',
        'paperless_echeck_risk',
        'high_risk_contract',
        'payment_type_grouped',
        'family_status'
    ],
    'Type': [
        'Binary Flag (0/1)',
        'Categorical (4 bins)',
        'Categorical (4 tiers)',
        'Discrete Count (0-9)',
        'Discrete Count (0-4)',
        'Binary Flag (0/1)',
        'Binary Flag (0/1)',
        'Binary Flag (0/1)',
        'Binary Flag (0/1)',
        'Categorical (3 groups)',
        'Categorical (4 groups)'
    ],
    'Business Purpose': [
        'Enables direct AVG/SUM aggregation in SQL & DAX',
        'Lifecycle cohort analysis (danger zone in Year 1)',
        'Pricing sensitivity & elasticity analysis',
        'Product holding depth / switching cost analysis',
        'Protection service adoption impact on retention',
        'Entertainment stickiness indicator',
        'Payment friction indicator',
        'High churn payment friction segment identifier',
        'Contract commitment vulnerability flag',
        'Simplified payment channel slicing in Power BI',
        'Household demographic churn analysis'
    ]
})

display(new_features_summary)

---

## Next Step

Proceed to **`04_segmentation.ipynb`** (Phase 6) where we:
1. Define the Multi-Factor Rule-Based Churn Risk Model
2. Assign customer risk segments (`High Risk`, `Medium Risk`, `Low Risk`)
3. Map tailored retention intervention strategies

---
**Feature Engineering Phase Complete.**